# Çözücüler ⚙️

Bu egzersizde, farklı `çözücülerin` `LogisticRegression` modelleri üzerindeki etkilerini araştıracaksınız.

👇 Veri kümesini içe aktarmak için aşağıdaki kodu çalıştırın

In [1]:
import pandas as pd

df = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/solvers_dataset.csv")
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,sulphates,alcohol,quality rating
0,9.47,5.97,7.36,10.17,6.84,9.15,9.78,9.52,10.34,8.80,6
1,10.05,8.84,9.76,8.38,10.15,6.91,9.70,9.01,9.23,8.80,7
2,10.59,10.71,10.84,10.97,9.03,10.42,11.46,11.25,11.34,9.06,4
3,11.00,8.44,8.32,9.65,7.87,10.92,6.97,11.07,10.66,8.89,8
4,12.12,13.44,10.35,9.95,11.09,9.38,10.22,9.04,7.68,11.38,3


- Veri kümesi farklı şaraplardan oluşmaktadır 🍷
- Özellikler şarapların farklı niteliklerini tanımlar 
- Hedef 🎯 bir uzman tarafından verilen kalite değerlendirmesidir

## 1. Hedef mühendisliği

Bu bölümde, değerlendirmeleri ikili bir hedefe dönüştüreceksiniz.

👇 Her değerlendirme için kaç gözlem bulunmaktadır?

In [4]:
df["quality rating"].value_counts().sort_index()

quality rating
1     10090
2     10030
3      9838
4      9928
5     10124
6      9961
7      9954
8      9977
9      9955
10    10143
Name: count, dtype: int64

❓ Hedefi ikili sınıflandırma görevine dönüştürerek `y` oluşturun, burada 6'nın altındaki kalite değerlendirmeleri kötü [0], 6 ve üzeri değerlendirmeler iyi [1] olacak

In [13]:
y = (df["quality rating"] >= 6).astype(int)
y

0        1
1        1
2        0
3        1
4        0
        ..
99995    1
99996    1
99997    0
99998    1
99999    0
Name: quality rating, Length: 100000, dtype: int64

❓ Yeni ikili hedefin sınıf dengesini kontrol edin

In [14]:
y.value_counts()

quality rating
0    50010
1    49990
Name: count, dtype: int64

In [22]:
y.value_counts(normalize=True)

quality rating
0    0.5001
1    0.4999
Name: proportion, dtype: float64

❓ Özellikleri normalleştirerek `X`'inizi oluşturun. Bu farklı çözücülerin adil karşılaştırılmasına olanak sağlayacaktır.

In [26]:
from sklearn.preprocessing import StandardScaler

x = df.drop(columns=["quality rating"])
scaler = StandardScaler()

scaler.fit_transform(x)

array([[-0.78860255, -1.52846058, -1.7331803 , ..., -0.47838741,
         0.34023114, -0.48983293],
       [-0.34686003, -0.46206949, -0.15828964, ..., -0.98697188,
        -0.76942881, -0.48983293],
       [ 0.06441748,  0.23275676,  0.55041116, ...,  1.24681087,
         1.33992478, -0.30738714],
       ...,
       [-0.20976753,  0.33307927,  1.14099516, ..., -0.1393311 ,
         0.04032304, -0.58807298],
       [-2.47941011, -2.27902156, -1.94972776, ...,  0.20969745,
        -1.67915003, -0.04073559],
       [ 2.06749131,  0.55973382, -0.84074225, ...,  0.96758803,
        -0.65946251,  0.14872736]], shape=(100000, 10))

## 2. LogisticRegression çözücüleri

❓ Lojistik Regresyon modelleri farklı **çözücüler** kullanılarak optimize edilebilir. Mevcut çözücülerin karşılaştırmasını yapın:
- Uyum süresi - hangi çözücü **en hızlı**?
- Kesinlik - kesinlik puanları **ne kadar farklı**?

Lojistik Regresyon için mevcut çözücüler: `['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga']`
 
Bu 5 çözücü hakkında daha fazla bilgi için [bu Stack Overflow konusuna](https://stackoverflow.com/questions/38640109/logistic-regression-python-solvers-defintions) göz atın

In [28]:
import time
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

solvers = ['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga']

for solver in solvers:
    model = LogisticRegression(solver=solver)
    
    start = time.time()
    model.fit(x_train, y_train)
    end = time.time()
    
    sure = round(end - start, 4)
    skor = round(model.score(x_test, y_test), 4)
    
    print(f"{solver} → Süre: {sure}s | Kesinlik: {skor}")

newton-cg → Süre: 0.129s | Kesinlik: 0.8659
lbfgs → Süre: 0.4005s | Kesinlik: 0.8659
liblinear → Süre: 0.2213s | Kesinlik: 0.8659
sag → Süre: 1.1039s | Kesinlik: 0.8659
saga → Süre: 0.1869s | Kesinlik: 0.8659


/home/demet/.pyenv/versions/3.12.9/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [30]:
# YOUR ANSWER
fastest_solver = "newton-cg"

<details>
    <summary>ℹ️ Yorumumuz için buraya tıklayın</summary>

Maliyet fonksiyonumuz 5 çözücünün de bulduğu global bir minimuma sahip olacak kadar "kolay" olduğundan, tüm çözücüler benzer kesinlik puanları üretmelidir. Derin Öğrenme'de olduğu gibi çok karmaşık maliyet fonksiyonları için, farklı çözücüler kayıp fonksiyonunun farklı değerlerinde durabilir.

**Şarap veri kümesi**
    
Mevcut veri kümesinde sklearn'in <a href="https://scikit-learn.org/stable/modules/generated/sklearn.inspection.permutation_importance.html">permutation_importance</a> ile özellik önemini kontrol ederseniz, birçok özelliğin neredeyse 0 önemine sahip olduğunu göreceksiniz. Liblinear çözücü, bir defada sadece *bir* yön boyunca hareket eder ve diğerlerini L1 düzenlileştirme ile düzenler (yani, beta değerlerini 0'a ayarlar), bu da birçok özelliğin hedefi tahmin etmede o kadar da önemli olmadığı bir veri kümesi için iyi bir uyum sağlayabilir.

❗️En iyi çözücüyü arama maliyeti vardır. Varsayılanla (`lbfgs`) devam etmek genel olarak en çok zaman tasarrufu sağlayabilir, sklearn başlamak için hangi çözücüyü seçeceğiniz konusunda fikir vermek için bu tabloyu sunar: 

<img src="https://wagon-public-datasets.s3.amazonaws.com/05-Machine-Learning/04-Under-the-Hood/solvers-chart.png" width=700>

</details>

###  🧪 Kodunuzu test edin

In [31]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'solvers',
    fastest_solver=fastest_solver
)
result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-9.0.2, pluggy-1.6.0 -- /home/demet/.pyenv/versions/3.12.9/bin/python3.12
cachedir: .pytest_cache
rootdir: /home/demet/S16D4-S-data-solvers/tests
plugins: anyio-4.12.1, dash-4.0.0
collecting ... collected 1 item

test_solvers.py::TestSolvers::test_fastest_solver PASSED                 [100%]

============================== 1 passed in 0.01s ===============================


💯 You can commit your code:

git add tests/solvers.pickle

git commit -m 'Completed solvers step'

git push origin master



## 3. Stokastik Gradyan İnişi

Lojistik Regresyon modelleri ayrıca Stokastik Gradyan İnişi ile de optimize edilebilir.

❓ **Stokastik Gradyan İnişi** ile optimize edilmiş bir Lojistik Regresyon modelini değerlendirin. Kesinlik puanı ve eğitim süresi 2. bölümde eğitilen modellerin performansı ile nasıl karşılaştırılır?

<details>
<summary>💡 İpucu</summary>

- Takılırsanız, [SGDClassifier belgelerine](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDClassifier.html) bakın!

</details>

In [32]:
from sklearn.linear_model import SGDClassifier

start = time.time()
sgd_model = SGDClassifier(loss='log_loss')
sgd_model.fit(x_train, y_train)
end = time.time()

sure = round(end - start, 4)
skor = round(sgd_model.score(x_test, y_test), 4)

print(f"SGD → Süre: {sure}s | Kesinlik: {skor}")

SGD → Süre: 0.8163s | Kesinlik: 0.8537


☝️ SGD modeli, benzer performans için en kısa sürelerden birine sahip olmalıdır (hatta `liblinear`'dan bile daha kısa olabilir). Bu, Gradyan İnişinin her dönemini aynı anda 100k satırı belleğe yüklemek yerine tek bir satırda gerçekleştirmenin doğrudan bir etkisidir.

## 4. Tahminler

❓ En iyi modeli (kısa uyum süresi ve yüksek kesinlik ile dengelenen) kullanarak aşağıdaki şarabın ikili kalitesini (0 veya 1) tahmin edin. Şunları kaydedin:
- `predicted_class`
- `predicted_proba_of_class` (yani modeliniz 1 sınıfını tahmin ettiyse, 1'in sınıf olması gerektiğine inanma olasılığı nedir, 0 ile 1 arasında olmalıdır)

In [33]:
new_wine = pd.read_csv('https://d32aokrjazspmn.cloudfront.net/materials/solvers_new_wine.csv')
new_wine

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,sulphates,alcohol
0,9.54,13.5,12.35,8.78,14.72,9.06,9.67,10.15,11.17,12.17


In [42]:

best_model = LogisticRegression(solver='newton-cg')
best_model.fit(x_train, y_train)


new_wine_scaled = scaler.transform(new_wine)


predicted_class = best_model.predict(new_wine_scaled)[0]
predicted_proba_of_class = best_model.predict_proba(new_wine_scaled)[0][predicted_class]

print(f"Tahmin edilen sınıf: {predicted_class}")
print(f"Olasılık: {predicted_proba_of_class}")

Tahmin edilen sınıf: 0
Olasılık: 0.7249520777132046


/home/demet/.pyenv/versions/3.12.9/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/home/demet/.pyenv/versions/3.12.9/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


# 🏁  Kodunuzu kontrol edin ve notebook'unuzu gönderin

In [43]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'new_data_prediction',
    predicted_class=predicted_class,
    predicted_proba_of_class=predicted_proba_of_class
)
result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-9.0.2, pluggy-1.6.0 -- /home/demet/.pyenv/versions/3.12.9/bin/python3.12
cachedir: .pytest_cache
rootdir: /home/demet/S16D4-S-data-solvers/tests
plugins: anyio-4.12.1, dash-4.0.0
collecting ... collected 2 items

test_new_data_prediction.py::TestNewDataPrediction::test_predicted_class PASSED [ 50%]
test_new_data_prediction.py::TestNewDataPrediction::test_predicted_proba FAILED [100%]

=================================== FAILURES ===================================
__________________ TestNewDataPrediction.test_predicted_proba __________________

self = <tests.test_new_data_prediction.TestNewDataPrediction testMethod=test_predicted_proba>

    def test_predicted_proba(self):
>       self.assertGreater(self.result.predicted_proba_of_class,0.93)
E       AssertionError: np.float64(0.7249520777132046) not greater than 0.93

test_new_data_prediction.py:9: AssertionError
==